# FINGUARD – GNN / GraphSAGE Analysis

Graph preparation, node features, GraphSAGE model definition, training, and evaluation.



In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

# ── Use original df (has nameOrig, nameDest) NOT sample_df ───────────────────
total_fraud  = df[df['isFraud'] == 1].shape[0]
total_normal = df[df['isFraud'] == 0].shape[0]
print(f"Available fraud rows:  {total_fraud}")
print(f"Available normal rows: {total_normal}")

n_fraud  = total_fraud
n_normal = min(total_normal, n_fraud * 4)

fraud_df_gnn  = df[df['isFraud'] == 1].sample(n_fraud,  random_state=42)
normal_df_gnn = df[df['isFraud'] == 0].sample(n_normal, random_state=42)

gnn_df = pd.concat([fraud_df_gnn, normal_df_gnn]).reset_index(drop=True)

print(f"\nGNN sample shape: {gnn_df.shape}")
print(f"Fraud: {gnn_df['isFraud'].sum()}  |  Normal: {(gnn_df['isFraud']==0).sum()}")


Encode node IDs and build edge list

In [ ]:
# ── 2. Encode account names → integer node IDs ───────────────────────────────
le = LabelEncoder()
all_accounts = pd.concat([gnn_df['nameOrig'], gnn_df['nameDest']])
le.fit(all_accounts)

src_nodes = le.transform(gnn_df['nameOrig'])   # sender node indices
dst_nodes = le.transform(gnn_df['nameDest'])   # receiver node indices

num_nodes = len(le.classes_)
print(f"Total unique accounts (nodes): {num_nodes}")
print(f"Total transactions (edges):    {len(gnn_df)}")

# Edge index tensor shape: [2, num_edges]
edge_index = torch.tensor([src_nodes, dst_nodes], dtype=torch.long)


Build node feature matrix

In [ ]:
# ── 3. Build node-level features ─────────────────────────────────────────────
# For each node (account), aggregate transaction features

# Tag each row with sender/receiver encoded ids
gnn_df['src_id'] = src_nodes
gnn_df['dst_id'] = dst_nodes

# Sender-side aggregation
sender_feat = gnn_df.groupby('src_id').agg(
    out_tx_count   = ('amount', 'count'),
    out_total_amt  = ('amount', 'sum'),
    out_avg_amt    = ('amount', 'mean'),
    out_max_amt    = ('amount', 'max'),
    fraud_sent     = ('isFraud', 'sum')
).reset_index().rename(columns={'src_id': 'node_id'})

# Receiver-side aggregation
receiver_feat = gnn_df.groupby('dst_id').agg(
    in_tx_count    = ('amount', 'count'),
    in_total_amt   = ('amount', 'sum'),
    in_avg_amt     = ('amount', 'mean'),
    in_max_amt     = ('amount', 'max'),
    fraud_received = ('isFraud', 'sum')
).reset_index().rename(columns={'dst_id': 'node_id'})

# Combine into one node feature table
node_df = pd.DataFrame({'node_id': range(num_nodes)})
node_df = node_df.merge(sender_feat,   on='node_id', how='left')
node_df = node_df.merge(receiver_feat, on='node_id', how='left')
node_df = node_df.fillna(0)

feature_cols = [
    'out_tx_count', 'out_total_amt', 'out_avg_amt', 'out_max_amt', 'fraud_sent',
    'in_tx_count',  'in_total_amt',  'in_avg_amt',  'in_max_amt',  'fraud_received'
]

scaler = StandardScaler()
X_nodes = scaler.fit_transform(node_df[feature_cols].values)
x = torch.tensor(X_nodes, dtype=torch.float)
print(f"Node feature matrix shape: {x.shape}")   # [num_nodes, 10]


In [ ]:
# ── 4. Build node-level fraud labels ─────────────────────────────────────────
# A node is labelled FRAUD if it appeared as sender in at least one fraud tx

node_labels = np.zeros(num_nodes, dtype=int)

fraud_senders = gnn_df[gnn_df['isFraud'] == 1]['src_id'].unique()
node_labels[fraud_senders] = 1

y = torch.tensor(node_labels, dtype=torch.long)
print(f"Fraud nodes:  {y.sum().item()}")
print(f"Normal nodes: {(y == 0).sum().item()}")


In [ ]:
# ── 5. Train / Test split as boolean masks ────────────────────────────────────
from sklearn.model_selection import train_test_split

indices = np.arange(num_nodes)
train_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=node_labels
)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask  = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[train_idx] = True
test_mask[test_idx]   = True

print(f"Train nodes: {train_mask.sum().item()}")
print(f"Test nodes:  {test_mask.sum().item()}")

# ── 6. Assemble PyG Data object ───────────────────────────────────────────────
data = Data(
    x          = x,
    edge_index = edge_index,
    y          = y,
    train_mask = train_mask,
    test_mask  = test_mask
)
print(data)


In [ ]:
# ── 7. Define GNN model ───────────────────────────────────────────────────────
# GraphSAGE: aggregates neighbour features — well-suited for fraud detection

class FraudGNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(FraudGNN, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.conv3 = SAGEConv(hidden_channels, out_channels)
        self.dropout = torch.nn.Dropout(p=0.3)

    def forward(self, x, edge_index):
        # Layer 1: message passing + ReLU
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)

        # Layer 2
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)

        # Layer 3: output logits
        x = self.conv3(x, edge_index)
        return x   # raw logits, shape [num_nodes, 2]

# Initialise model
model = FraudGNN(
    in_channels     = x.shape[1],   # 10 features
    hidden_channels = 64,
    out_channels    = 2             # 0 = normal, 1 = fraud
)

# Class weights to handle imbalance (more normal nodes than fraud)
fraud_weight  = (y == 0).sum().float() / (y == 1).sum().float()
class_weights = torch.tensor([1.0, fraud_weight])

optimizer  = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion  = torch.nn.CrossEntropyLoss(weight=class_weights)

print(model)
print(f"\nClass weight for fraud nodes: {fraud_weight:.2f}")


In [ ]:
# ── 8. Training loop ──────────────────────────────────────────────────────────
def train():
    model.train()
    optimizer.zero_grad()
    out  = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def evaluate(mask):
    model.eval()
    out   = model(data.x, data.edge_index)
    pred  = out.argmax(dim=1)
    correct = (pred[mask] == data.y[mask]).sum().item()
    total   = mask.sum().item()
    return correct / total

# Run training
history = {'loss': [], 'train_acc': [], 'test_acc': []}

for epoch in range(1, 151):
    loss      = train()
    train_acc = evaluate(data.train_mask)
    test_acc  = evaluate(data.test_mask)

    history['loss'].append(loss)
    history['train_acc'].append(train_acc)
    history['test_acc'].append(test_acc)

    if epoch % 25 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss:.4f} | "
              f"Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")

print("\nTraining complete!")


Plot training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(history['loss'], color='crimson', linewidth=2)
axes[0].set_title("GNN Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(history['train_acc'], label='Train Accuracy', color='steelblue', linewidth=2)
axes[1].plot(history['test_acc'],  label='Test Accuracy',  color='darkorange', linewidth=2)
axes[1].set_title("GNN Node Classification Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


Full evaluation (precision, recall, F1)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

model.eval()
with torch.no_grad():
    out        = model(data.x, data.edge_index)
    pred_all   = out.argmax(dim=1)

# Extract test predictions
y_true_gnn = data.y[data.test_mask].numpy()
y_pred_gnn = pred_all[data.test_mask].numpy()

print("=" * 50)
print("GNN Fraud Detection — Classification Report")
print("=" * 50)
print(classification_report(y_true_gnn, y_pred_gnn, target_names=['Normal', 'Fraud']))

# Confusion matrix
cm_gnn = confusion_matrix(y_true_gnn, y_pred_gnn)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_gnn, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Normal', 'Fraud'],
            yticklabels=['Normal', 'Fraud'])
plt.title("GNN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


Visualize fraud node embeddings with t-SNE

In [ ]:
from sklearn.manifold import TSNE

# Get intermediate embeddings from layer 2 (before final classification)
class EmbeddingGNN(torch.nn.Module):
    def __init__(self, trained_model):
        super().__init__()
        self.conv1 = trained_model.conv1
        self.conv2 = trained_model.conv2
        self.dropout = trained_model.dropout

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv2(x, edge_index))
        return x

embed_model = EmbeddingGNN(model)
embed_model.eval()

with torch.no_grad():
    embeddings = embed_model(data.x, data.edge_index).numpy()

# t-SNE: reduce 64-dim embeddings to 2D for plotting
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
emb_2d = tsne.fit_transform(embeddings)

# Plot
labels_np = data.y.numpy()
colors = ['steelblue' if l == 0 else 'crimson' for l in labels_np]
plt.figure(figsize=(10, 7))
scatter = plt.scatter(emb_2d[:, 0], emb_2d[:, 1],
                      c=labels_np, cmap='coolwarm',
                      alpha=0.5, s=5,        # smaller s
                      edgecolors='none',
                      linewidths=0)           # no connecting lines
plt.colorbar(scatter, label='0 = Normal | 1 = Fraud')
plt.title("t-SNE of GNN Node Embeddings\n(Fraud vs Normal Accounts)")
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.tight_layout()
plt.show()


Compare GNN vs your ML models

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

# Add GNN scores to your existing comparison table
gnn_precision = precision_score(y_true_gnn, y_pred_gnn)
gnn_recall    = recall_score(y_true_gnn, y_pred_gnn)
gnn_f1        = f1_score(y_true_gnn, y_pred_gnn)

# Build comparison (reuse your earlier y_pred_rf, y_pred_dt, y_pred_lr)
final_comparison = pd.DataFrame({
    "Model":     ["Logistic Regression", "Decision Tree", "Random Forest","XGBoost","LightGBM", "GNN (GraphSAGE)"],
    "Precision": [precision_score(y_test, y_pred_lr),
                  precision_score(y_test, y_pred_dt),
                  precision_score(y_test, y_pred_rf),
                  precision_score(y_test, y_pred_xgb),
                  precision_score(y_test, y_pred_lgbm),
                  gnn_precision],
    "Recall":    [recall_score(y_test, y_pred_lr),
                  recall_score(y_test, y_pred_dt),
                  recall_score(y_test, y_pred_rf),
                  recall_score(y_test, y_pred_xgb),
                  recall_score(y_test, y_pred_lgbm),
                  gnn_recall],
    "F1 Score":  [f1_score(y_test, y_pred_lr),
                  f1_score(y_test, y_pred_dt),
                  f1_score(y_test, y_pred_rf),
                  f1_score(y_test, y_pred_xgb),
                  f1_score(y_test, y_pred_lgbm),
                  gnn_f1]
})

print(final_comparison.to_string(index=False))

# Plot
comparison_melt = final_comparison.melt(id_vars="Model", var_name="Metric", value_name="Score")
plt.figure(figsize=(12, 6))
sns.barplot(data=comparison_melt, x="Model", y="Score", hue="Metric", palette="viridis")
plt.title("Model Comparison: ML vs GNN")
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()


In [ ]:
model.eval()
with torch.no_grad():
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)

train_acc = (pred[data.train_mask] == data.y[data.train_mask]).sum().item() / data.train_mask.sum().item()
test_acc  = (pred[data.test_mask]  == data.y[data.test_mask]).sum().item()  / data.test_mask.sum().item()

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test  Accuracy: {test_acc:.4f}")

if abs(train_acc - test_acc) < 0.02:
    print("\nVerdict: NOT overfitting — train and test scores are close")
else:
    print("\nVerdict: Possible overfitting — large gap between train and test")


ANOMALY DETECTION
¶

ISOLATION FOREST

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ── Find actual fraud percentage ──────────────────────────────────
fraud_pct = y_test.sum() / len(y_test)
print(f"Actual fraud percentage: {fraud_pct:.6f}")

# ── Train Isolation Forest ────────────────────────────────────────
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=fraud_pct,   # ← changed from 0.002 to actual %
    random_state=42
)
iso_forest.fit(X_test)

# ── Predict (-1 = anomaly/fraud, 1 = normal) ─────────────────────
iso_pred_raw = iso_forest.predict(X_test)
y_pred_iso   = np.where(iso_pred_raw == -1, 1, 0)

print("=" * 50)
print("  Isolation Forest Results")
print("=" * 50)
print(f"Anomalies detected: {y_pred_iso.sum()}")
print(f"Actual fraud count: {y_test.sum()}")
print(classification_report(y_test, y_pred_iso,
      target_names=['Normal', 'Fraud']))


In [ ]:
# ISLOATION FOREST CONFUSION MATRIX
plt.figure(figsize=(6, 5))
cm_iso = confusion_matrix(y_test, y_pred_iso)
sns.heatmap(cm_iso, annot=True, fmt='d',
            cmap='RdPu',
            xticklabels=['Normal', 'Fraud'],
            yticklabels=['Normal', 'Fraud'],
            linewidths=0.5,
            linecolor='white')

plt.title("Isolation Forest Confusion Matrix",
          fontsize=13, fontweight='bold', color='#2d2d2d', pad=12)
plt.xlabel("Predicted", fontsize=11, color='#444444')
plt.ylabel("Actual", fontsize=11, color='#444444')
plt.gcf().set_facecolor("white")
plt.tight_layout()
plt.show()


ANOMALY SCORE DISTRIBUTION
¶

In [ ]:
# Get anomaly scores
anomaly_scores = iso_forest.decision_function(X_test)

# More negative = more anomalous
plt.figure(figsize=(12, 5))

ax = plt.gca()
ax.set_facecolor("white")
plt.gcf().set_facecolor("white")

# Plot distribution for normal and fraud separately
normal_scores = anomaly_scores[y_test == 0]
fraud_scores  = anomaly_scores[y_test == 1]

plt.hist(normal_scores, bins=50, alpha=0.7,
         color='#7b2d8b', label='Normal', edgecolor='white')
plt.hist(fraud_scores,  bins=50, alpha=0.7,
         color='#f5b048', label='Fraud',  edgecolor='white')

plt.axvline(x=0, color='#c0392b', linestyle='--',
            linewidth=1.5, label='Decision boundary')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_color('#dddddd')
ax.spines['left'].set_color('#dddddd')
ax.tick_params(colors='#444444', labelsize=10)
ax.yaxis.grid(True, color='#eeeeee', linewidth=0.8)
ax.set_axisbelow(True)

plt.title("Isolation Forest — Anomaly Score Distribution",
          fontsize=14, fontweight='bold', color='#2d2d2d', pad=15)
plt.xlabel("Anomaly Score (more negative = more suspicious)",
           fontsize=11, color='#444444')
plt.ylabel("Number of Transactions", fontsize=11, color='#444444')
plt.legend(fontsize=10, facecolor='white', edgecolor='#dddddd')
plt.tight_layout()
plt.show()


Compare Supervised vs Unsupervised
¶

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

# Compare all models including Isolation Forest
compare_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree",
              "Random Forest", "LightGBM",
              "Isolation Forest (Unsupervised)"],
    "Precision": [precision_score(y_test, y_pred_lr),
                  precision_score(y_test, y_pred_dt),
                  precision_score(y_test, y_pred_rf),
                  precision_score(y_test, y_pred_lgbm),
                  precision_score(y_test, y_pred_iso)],
    "Recall":    [recall_score(y_test, y_pred_lr),
                  recall_score(y_test, y_pred_dt),
                  recall_score(y_test, y_pred_rf),
                  recall_score(y_test, y_pred_lgbm),
                  recall_score(y_test, y_pred_iso)],
    "F1 Score":  [f1_score(y_test, y_pred_lr),
                  f1_score(y_test, y_pred_dt),
                  f1_score(y_test, y_pred_rf),
                  f1_score(y_test, y_pred_lgbm),
                  f1_score(y_test, y_pred_iso)]
})

print(compare_df.to_string(index=False))

# ── Plot ──────────────────────────────────────────────────────────
compare_melt = compare_df.melt(
    id_vars="Model", var_name="Metric", value_name="Score"
)

plt.figure(figsize=(14, 6))
ax = sns.barplot(
    data=compare_melt,
    x="Model", y="Score",
    hue="Metric", palette="RdPu"
)
ax.set_facecolor("white")
plt.gcf().set_facecolor("white")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_color('#dddddd')
ax.spines['left'].set_color('#dddddd')
ax.tick_params(colors='#444444', labelsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=15)
ax.set_ylim(0, 1.1)
ax.yaxis.grid(True, color='#eeeeee', linewidth=0.8)
ax.set_axisbelow(True)

for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3,
                 fontsize=7, color='#333333')

plt.title("Supervised vs Unsupervised — Fraud Detection Comparison",
          fontsize=13, fontweight='bold', color='#2d2d2d', pad=15)
plt.xlabel("Model", fontsize=11, color='#444444')
plt.ylabel("Score", fontsize=11, color='#444444')
plt.legend(title="Metric", fontsize=9,
           facecolor='white', edgecolor='#dddddd')
plt.tight_layout()
plt.show()


In [ ]:
import pickle
import json

# Save Random Forest model
with open('rf_model.pkl', 'wb') as f:
    pickle.dump(rf, f)

# Save LightGBM model
with open('lgbm_model.pkl', 'wb') as f:
    pickle.dump(lgbm, f)

# Save feature columns
with open('feature_columns.json', 'w') as f:
    json.dump(list(X_test.columns), f)

print("✅ Models saved successfully!")
print("📁 Files created:")
print("   - rf_model.pkl")
print("   - lgbm_model.pkl")
print("   - feature_columns.json")


In [ ]:
import os
print("Files saved in this folder:")
print(os.getcwd())
print("\nFiles in this folder:")
for f in os.listdir():
    print(f)


In [ ]:
import pickle
import numpy as np
import json

# Save GNN results
np.save('gnn_emb_2d.npy', emb_2d)
np.save('gnn_labels.npy', labels_np)
np.save('gnn_y_true.npy', y_true_gnn)
np.save('gnn_y_pred.npy', y_pred_gnn)

with open('gnn_history.json', 'w') as f:
    json.dump(history, f)

print("✅ GNN results saved!")
